In [ ]:
# Import required libraries
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

# Add path to modules
sys.path.insert(1, r'C:\Zmyfiles\codes\research\KPCA-DA\KPCA-DA\examples\Scripts')

# Import the domain adaptive image transformer
from pyDeepInsight.image_transformer_new import DomainAdaptiveImageTransformer

## Load Source and Target Datasets

Load gene expression data for both source (TCGA) and target (GSE45498/DESMD) datasets.

In [ ]:
# Load source dataset (TCGA)
expr_file_source = r"..\Data\TvsN\TCGA_BRCA_NvT.csv"
expr_source = pd.read_csv(expr_file_source, encoding='utf-8', engine='python')
expr_source = expr_source.transpose()
expr_source.columns = expr_source.iloc[0]

X_source = expr_source.iloc[1:, 0:-1].values.astype(float)
y_source = expr_source.iloc[1:, -1].values
genes_source = expr_source.iloc[:, :-1].columns.to_numpy()

print(f"Source dataset shape: {X_source.shape}")
print(f"Source genes: {len(genes_source)}")
print(f"Source samples: {X_source.shape[0]}")

In [ ]:
# Load target dataset (GSE45498/DESMD)
expr_file_target = r"..\Data\TvsN\TCGA_BRCA_NvTv1_filtered.csv"
expr_target = pd.read_csv(expr_file_target, encoding='utf-8', engine='python')
expr_target = expr_target.transpose()
expr_target.columns = expr_target.iloc[0]

X_target = expr_target.iloc[1:, 0:-1].values.astype(float)
y_target = expr_target.iloc[1:, -1].values
genes_target = expr_target.iloc[:, :-1].columns.to_numpy()

print(f"Target dataset shape: {X_target.shape}")
print(f"Target genes: {len(genes_target)}")
print(f"Target samples: {X_target.shape[0]}")

## Data Preprocessing

Normalize the data and encode labels.

In [ ]:
# Normalize the data
mms = MinMaxScaler()
X_source_norm = mms.fit_transform(X_source)
X_target_norm = mms.fit_transform(X_target)

# Encode labels
le = LabelEncoder()
y_source_enc = le.fit_transform(y_source)
y_target_enc = le.transform(y_target)

le_mapping = dict(zip(le.transform(le.classes_), le.classes_))
num_classes = np.unique(y_source_enc).size

print(f"Number of classes: {num_classes}")
print(f"Class mapping: {le_mapping}")
print(f"Source label distribution: {np.bincount(y_source_enc)}")
print(f"Target label distribution: {np.bincount(y_target_enc)}")

## Initialize Domain Adaptive Image Transformer

Create the transformer with KPCA-DA parameters for domain adaptation.

In [ ]:
# Configure KPCA-DA parameters
kpca_da_params = {
    'n_components': 2,        # Number of KPCA components
    'kernel': 'rbf',          # Kernel type
    'kpca_gamma': 0.2,        # Gamma for KPCA kernel
    'mmd_gamma': 0.5,         # Gamma for MMD kernel
    'lambda_reg': 0.1,        # Regularization strength
    'lr': 0.0001,             # Learning rate
    'epochs': 100,            # Number of optimization epochs
    'verbose': True           # Print progress
}

# Initialize the domain adaptive image transformer
pixel_size = (227, 227)

da_transformer = DomainAdaptiveImageTransformer(
    kpca_da_params=kpca_da_params,
    discretization='bin',
    pixels=pixel_size
)

print(f"Initialized DomainAdaptiveImageTransformer with pixel size: {pixel_size}")

## Fit and Transform

Apply KPCA-DA adaptation and create images for both source and target datasets.

In [ ]:
# Fit and transform both datasets
print("Fitting transformer and creating images...")
print("="*70)

X_source_img, X_target_img = da_transformer.fit_transform(
    X_source=X_source_norm,
    X_target=X_target_norm,
    genes_source=genes_source,
    genes_target=genes_target,
    plot=True,  # Show visualization of adaptation
    img_format='rgb',
    empty_value=0
)

print("\n" + "="*70)
print("Image transformation complete!")
print(f"Source images shape: {X_source_img.shape}")
print(f"Target images shape: {X_target_img.shape}")

## Visualize Results

Show sample images from both source and target datasets.

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Show 4 source images
for i in range(4):
    axes[0, i].imshow(X_source_img[i])
    axes[0, i].set_title(f'Source Sample {i+1}\nClass: {le_mapping[y_source_enc[i]]}', 
                        fontsize=10, fontweight='bold')
    axes[0, i].axis('off')

# Show 4 target images
for i in range(4):
    axes[1, i].imshow(X_target_img[i])
    axes[1, i].set_title(f'Target Sample {i+1}\nClass: {le_mapping[y_target_enc[i]]}', 
                        fontsize=10, fontweight='bold')
    axes[1, i].axis('off')

plt.suptitle('Sample Images: Source (Top) vs Target (Bottom) after KPCA-DA Adaptation', 
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## Feature Density Analysis

Visualize how features are distributed across pixels in both datasets.

In [ ]:
# Get feature density matrices
density_source = da_transformer.feature_density_matrix_source()
density_target = da_transformer.feature_density_matrix_target()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Source feature density
im1 = axes[0].imshow(density_source, cmap='hot', interpolation='nearest')
axes[0].set_title('Source Dataset Feature Density', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Pixel X')
axes[0].set_ylabel('Pixel Y')
plt.colorbar(im1, ax=axes[0], label='Feature Count')

# Target feature density
im2 = axes[1].imshow(density_target, cmap='hot', interpolation='nearest')
axes[1].set_title('Target Dataset Feature Density', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Pixel X')
axes[1].set_ylabel('Pixel Y')
plt.colorbar(im2, ax=axes[1], label='Feature Count')

plt.tight_layout()
plt.show()

print(f"Source max features per pixel: {int(density_source.max())}")
print(f"Target max features per pixel: {int(density_target.max())}")
print(f"Source total mapped features: {int(density_source.sum())}")
print(f"Target total mapped features: {int(density_target.sum())}")

## KPCA-DA Loss History

Plot the optimization loss during domain adaptation.

In [ ]:
# Plot loss history
loss_history = da_transformer.loss_history

plt.figure(figsize=(10, 6))
plt.plot(loss_history, linewidth=2, color='darkblue')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Total Loss (MMD + Regularization)', fontsize=12)
plt.title('KPCA-DA Optimization Loss History', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Initial loss: {loss_history[0]:.6f}")
print(f"Final loss: {loss_history[-1]:.6f}")
print(f"Loss reduction: {((loss_history[0] - loss_history[-1]) / loss_history[0] * 100):.2f}%")

## Summary

The `DomainAdaptiveImageTransformer` successfully:
1. Applied KPCA-DA to align target domain to source domain
2. Created unified coordinate system for both datasets
3. Generated image representations preserving spatial relationships
4. Enabled both datasets to be used in the same CNN pipeline

The adapted images can now be used for training models on source data and testing on target data.

In [ ]:
# Print summary statistics
print("="*70)
print("DOMAIN ADAPTIVE IMAGE TRANSFORMER SUMMARY")
print("="*70)
print(f"Source samples:           {X_source_img.shape[0]}")
print(f"Target samples:           {X_target_img.shape[0]}")
print(f"Image dimensions:         {X_source_img.shape[1:3]}")
print(f"Common genes found:       {da_transformer.kpca_da.common_source_idx_.shape[0]}")
print(f"KPCA components:          {da_transformer.kpca_da.n_components}")
print(f"Final adaptation loss:    {loss_history[-1]:.6f}")
print(f"Pixel utilization (src):  {(density_source > 0).sum() / np.prod(pixel_size) * 100:.1f}%")
print(f"Pixel utilization (tgt):  {(density_target > 0).sum() / np.prod(pixel_size) * 100:.1f}%")
print("="*70)